# Renger 2026 Figure 2(e)(f) Retune

이 notebook은 baseline Renger snapshot 위에서 `shared beta_cr + branch-specific TC/beta_qc/t_gate` 를 좁혀서, Fig. 2(e)(f)와 비슷한 basin/stripe morphology를 주는 후보를 **shortlist + visual pick** 방식으로 찾기 위한 working harness입니다.

- baseline TOML은 건드리지 않습니다.
- 결과는 `output/renger2026/fig2_ef_retune_working.toml` overlay로만 저장합니다.
- 자동 score는 generic physics 기준만 쓰고, 최종 선택은 gallery를 보고 사람이 고릅니다.


## Outline

1. Activate the local environment and load the shared Fig. 2 branch helpers.
2. Load the baseline snapshot and the current working overlay.
3. Define deterministic coarse/fine retune helpers with a compact default profile.
4. Build a shared-`beta_cr` joint shortlist for MOVE and CZ.
5. Support a `:focused_full` rerun around the current overlay checkpoint.
6. Render a paper-span-centered gallery for the top candidates.
7. Optionally refine the selected candidate and write it back to the working overlay.


In [ ]:
using Pkg

function find_repo_root(start::AbstractString)
    candidates = [
        normpath(start),
        normpath(joinpath(start, "..")),
        normpath(joinpath(start, "..", "..")),
    ]

    for candidate in unique(candidates)
        project_toml = joinpath(candidate, "Project.toml")
        if isfile(project_toml)
            content = read(project_toml, String)
            occursin("QuantumCircuit", content) && return candidate
        end
    end

    error("Could not find the QuantumCircuit.jl repository root from $(start).")
end

repo_root = find_repo_root(pwd())
Pkg.activate(repo_root)
Pkg.instantiate()

using Printf
using TOML
using CairoMakie
using QuantumCircuit
using QuantumToolbox: basis, dag, eigenstates, kron, Ket, QuantumObject

include(joinpath(repo_root, "output", "jupyter-notebook", "makie_helpers.jl"))
include(joinpath(repo_root, "output", "jupyter-notebook", "renger2026_fig2_branch_helpers.jl"))
activate_notebook_theme!()

const RETUNE_BASELINE_PATH = joinpath(repo_root, "output", "renger2026", "paper_local_priors.toml")
const RETUNE_OVERLAY_PATH = joinpath(repo_root, "output", "renger2026", "fig2_ef_retune_working.toml")
const RETUNE_PROFILE = :compact
const RETUNE_CHARGE_CUTOFF = 3
const RETUNE_MAX_ROUNDS = 2
const RETUNE_SHORTLIST_SIZE = 6
const RETUNE_COARSE_Q_POINTS = 9
const RETUNE_COARSE_TC_POINTS = 9
const RETUNE_FINE_Q_POINTS = 21
const RETUNE_FINE_TC_POINTS = 21
const MOVE_SPAN = (delta_q_ghz = 0.10, delta_tc_ghz = 1.60)
const CZ_SPAN = (delta_q_ghz = 0.11, delta_tc_ghz = 1.90)
const SELECT_CANDIDATE_RANK = 1
const RUN_REFINEMENT = true
const RUN_EC_FALLBACK = false
const WRITE_WORKING_OVERLAY = false
const FIG2_DEFAULT_MOVE_T_GATE = 100.0
const FIG2_DEFAULT_CZ_T_GATE = 120.0
retune_exports = Dict{Symbol, NamedTuple}()

nothing


## Baseline And Working Overlay

The retune starts from the baseline local-prior snapshot and then overlays the current working file if it exists. The working file is the only persistent location this notebook writes to.


In [ ]:
baseline_snapshot = load_renger2026_snapshot(RETUNE_BASELINE_PATH)
working_snapshot = load_renger2026_snapshot(
    RETUNE_BASELINE_PATH;
    overlay_path = isfile(RETUNE_OVERLAY_PATH) ? RETUNE_OVERLAY_PATH : nothing,
)
working_gate_times = fig2_gate_times(
    working_snapshot;
    move_default = FIG2_DEFAULT_MOVE_T_GATE,
    cz_default = FIG2_DEFAULT_CZ_T_GATE,
)
parameter_summary, coupling_summary = parameter_rows(
    working_snapshot;
    move_t_gate_ns = working_gate_times.move,
    cz_t_gate_ns = working_gate_times.cz,
    charge_cutoff = RETUNE_CHARGE_CUTOFF,
)

(
    baseline_path = RETUNE_BASELINE_PATH,
    overlay_path = isfile(RETUNE_OVERLAY_PATH) ? RETUNE_OVERLAY_PATH : missing,
    profile = RETUNE_PROFILE,
    parameter_summary = parameter_summary,
    coupling_summary = coupling_summary,
)


## Retune Helpers

The coarse pass keeps `EC` fixed at the current working value and searches over `EJmax`, `asymmetry`, branch-local `beta_qc`, shared `beta_cr`, panel `t_gate`, and `tc_flux_offset`. Ranking is explicitly interior-first, and the `:focused_full` profile re-centers the search around the current overlay checkpoint while keeping the same scoring logic. The baseline CR target is the Table II paper-exact `4.22 GHz`.


In [ ]:
const FULL_BETA_CR_GRID = collect(0.016:0.002:0.032)
const FULL_EJMAX_GRID = collect(28.0:2.0:44.0)
const FULL_ASYM_GRID = collect(0.04:0.02:0.14)
const FULL_TC_FLUX_OFFSET_GRID = collect(-0.04:0.02:0.12)
const FULL_MOVE_BETA_QC_GRID = collect(0.013:0.002:0.023)
const FULL_CZ_BETA_QC_GRID = collect(0.018:0.002:0.030)
const FULL_T_GATE_GRID = collect(60.0:20.0:180.0)
const FALLBACK_EC_GRID = collect(0.095:0.005:0.120)

const BRANCH_SCORE_CACHE = Dict{Tuple, Any}()
const JOINT_CANDIDATE_CACHE = Dict{Tuple, Any}()

function search_profile_settings(profile::Symbol)
    profile == :full && return (
        beta_cr_count = length(FULL_BETA_CR_GRID),
        branch_grid_count = 0,
        offset_grid_count = 0,
        branch_keep = 3,
        seeds_per_slice = 2,
        multistart = 3,
        seed_pool = 12,
    )
    profile == :focused_full && return (
        beta_cr_count = 5,
        branch_grid_count = 3,
        offset_grid_count = 3,
        branch_keep = 4,
        seeds_per_slice = 2,
        multistart = 3,
        seed_pool = 12,
    )
    profile == :compact && return (
        beta_cr_count = 3,
        branch_grid_count = 2,
        offset_grid_count = 3,
        branch_keep = 3,
        seeds_per_slice = 1,
        multistart = 2,
        seed_pool = 8,
    )
    profile == :tiny && return (
        beta_cr_count = 1,
        branch_grid_count = 1,
        offset_grid_count = 1,
        branch_keep = 1,
        seeds_per_slice = 1,
        multistart = 1,
        seed_pool = 4,
    )
    throw(ArgumentError("RETUNE_PROFILE must be :compact, :focused_full, :full, or :tiny, got $(profile)."))
end

branch_tc_key(branch::Symbol) = branch_spec(branch).tc_key
branch_beta_qc_key(branch::Symbol) = branch_spec(branch).beta_qc_key
branch_init_levels(branch::Symbol) = branch == :move ? (1, 0, 0) : (1, 0, 1)
branch_pairs(branch::Symbol) = branch == :move ? ((1, 0, 0), (0, 0, 1)) : ((1, 0, 1), (2, 0, 0))
branch_default_t_gate(branch::Symbol) = branch == :move ? FIG2_DEFAULT_MOVE_T_GATE : FIG2_DEFAULT_CZ_T_GATE
branch_span(branch::Symbol) = branch == :move ? MOVE_SPAN : CZ_SPAN

retune_profile_suffix(profile::Symbol) = profile == :compact ? "" : "_" * String(profile)

function retune_artifact_stem(base::AbstractString; profile::Symbol = RETUNE_PROFILE)
    suffix = retune_profile_suffix(profile)
    return isempty(suffix) ? base : base * suffix
end

function branch_flux_ranges(branch::Symbol)
    branch == :move && return ((0.18, 0.26), (0.12, 0.40))
    return ((0.12, 0.24), (0.12, 0.40))
end

function branch_flux_axes(branch::Symbol; q_points, tc_points)
    (q_lo, q_hi), (tc_lo, tc_hi) = branch_flux_ranges(branch)
    return (
        q_fluxes = collect(range(q_lo, q_hi; length = q_points)),
        tc_fluxes = collect(range(tc_lo, tc_hi; length = tc_points)),
    )
end

function centered_subset(values::AbstractVector, baseline; count::Integer)
    length(values) <= count && return collect(values)
    perm = sortperm(eachindex(values); by = i -> (abs(Float64(values[i]) - baseline), i))
    idxs = sort(collect(perm[1:count]))
    return collect(values[idxs])
end

function beta_cr_grid(snapshot, profile)
    settings = search_profile_settings(profile)
    profile == :full && return FULL_BETA_CR_GRID
    if profile == :focused_full
        focused = collect(0.026:0.001:0.030)
        return centered_subset(focused, snapshot.targets["beta_cr"]; count = settings.beta_cr_count)
    end
    return centered_subset(FULL_BETA_CR_GRID, snapshot.targets["beta_cr"]; count = settings.beta_cr_count)
end

function branch_grids(snapshot, gate_times, branch::Symbol, profile)
    beta_qc_full = branch == :move ? FULL_MOVE_BETA_QC_GRID : FULL_CZ_BETA_QC_GRID
    tc_key = branch_tc_key(branch)
    baseline_t_gate = branch == :move ? gate_times.move : gate_times.cz
    settings = search_profile_settings(profile)
    if profile == :full
        return (
            EJmax = FULL_EJMAX_GRID,
            asymmetry = FULL_ASYM_GRID,
            beta_qc = beta_qc_full,
            t_gate = FULL_T_GATE_GRID,
            tc_flux_offset = FULL_TC_FLUX_OFFSET_GRID,
        )
    end
    if profile == :focused_full
        focused = branch == :move ? (
            EJmax = collect(34.0:0.5:38.0),
            asymmetry = collect(0.08:0.01:0.14),
            beta_qc = collect(0.019:0.001:0.023),
            t_gate = collect(60.0:10.0:100.0),
            tc_flux_offset = collect(0.00:0.01:0.06),
        ) : (
            EJmax = collect(32.0:0.5:36.0),
            asymmetry = collect(0.04:0.01:0.08),
            beta_qc = collect(0.024:0.001:0.030),
            t_gate = collect(80.0:10.0:120.0),
            tc_flux_offset = collect(0.00:0.01:0.06),
        )
        return (
            EJmax = centered_subset(focused.EJmax, snapshot.devices[tc_key]["EJmax"]; count = settings.branch_grid_count),
            asymmetry = centered_subset(focused.asymmetry, snapshot.devices[tc_key]["asymmetry"]; count = settings.branch_grid_count),
            beta_qc = centered_subset(focused.beta_qc, snapshot.targets[branch_beta_qc_key(branch)]; count = settings.branch_grid_count),
            t_gate = centered_subset(focused.t_gate, baseline_t_gate; count = settings.branch_grid_count),
            tc_flux_offset = centered_subset(focused.tc_flux_offset, snapshot.devices[tc_key]["flux"]; count = settings.offset_grid_count),
        )
    end
    return (
        EJmax = centered_subset(FULL_EJMAX_GRID, snapshot.devices[tc_key]["EJmax"]; count = settings.branch_grid_count),
        asymmetry = centered_subset(FULL_ASYM_GRID, snapshot.devices[tc_key]["asymmetry"]; count = settings.branch_grid_count),
        beta_qc = centered_subset(beta_qc_full, snapshot.targets[branch_beta_qc_key(branch)]; count = settings.branch_grid_count),
        t_gate = centered_subset(FULL_T_GATE_GRID, baseline_t_gate; count = settings.branch_grid_count),
        tc_flux_offset = centered_subset(FULL_TC_FLUX_OFFSET_GRID, snapshot.devices[tc_key]["flux"]; count = settings.offset_grid_count),
    )
end

function copy_branch_params(params::AbstractDict{String, <:Any})
    return Dict(key => value for (key, value) in params)
end

function copy_joint_params(params::Dict{String, Any})
    return Dict(
        "beta_cr" => params["beta_cr"],
        "move" => copy_branch_params(params["move"]),
        "cz" => copy_branch_params(params["cz"]),
    )
end

function branch_bounds(branch::Symbol)
    beta_qc_full = branch == :move ? FULL_MOVE_BETA_QC_GRID : FULL_CZ_BETA_QC_GRID
    return Dict(
        "EJmax" => extrema(FULL_EJMAX_GRID),
        "EC" => extrema(FALLBACK_EC_GRID),
        "asymmetry" => extrema(FULL_ASYM_GRID),
        "beta_qc" => extrema(beta_qc_full),
        "t_gate" => extrema(FULL_T_GATE_GRID),
        "tc_flux_offset" => extrema(FULL_TC_FLUX_OFFSET_GRID),
    )
end

function clamp_branch_param(branch::Symbol, field::String, value)
    lo, hi = branch_bounds(branch)[field]
    return clamp(Float64(value), lo, hi)
end

function clamp_beta_cr(value)
    lo, hi = extrema(FULL_BETA_CR_GRID)
    return clamp(Float64(value), lo, hi)
end

function branch_candidate_snapshot(base_snapshot, branch::Symbol, params::AbstractDict{String, <:Any})
    tc_key = branch_tc_key(branch)
    device_overrides = Dict(
        tc_key => Dict(
            "EJmax" => params["EJmax"],
            "EC" => params["EC"],
            "flux" => params["tc_flux_offset"],
            "asymmetry" => params["asymmetry"],
        ),
    )
    gate_time_key = branch == :move ? "fig2_move_t_gate_ns" : "fig2_cz_t_gate_ns"
    target_overrides = Dict(
        branch_beta_qc_key(branch) => params["beta_qc"],
        "beta_cr" => params["beta_cr"],
        gate_time_key => params["t_gate"],
    )
    return snapshot_with_overrides(base_snapshot; device_overrides, target_overrides)
end

function branch_score_cache_key(branch::Symbol, params::AbstractDict{String, <:Any}; charge_cutoff, q_points, tc_points, max_rounds)
    return (
        branch,
        round(Int, params["EJmax"] * 1000),
        round(Int, params["EC"] * 1_000_000),
        round(Int, params["asymmetry"] * 10_000),
        round(Int, params["beta_qc"] * 10_000),
        round(Int, params["beta_cr"] * 10_000),
        round(Int, params["t_gate"] * 100),
        round(Int, params["tc_flux_offset"] * 10_000),
        charge_cutoff,
        q_points,
        tc_points,
        max_rounds,
    )
end

function branch_score_summary(base_snapshot, branch::Symbol, params::AbstractDict{String, <:Any}; charge_cutoff = RETUNE_CHARGE_CUTOFF, q_points = RETUNE_COARSE_Q_POINTS, tc_points = RETUNE_COARSE_TC_POINTS, max_rounds = RETUNE_MAX_ROUNDS)
    cache_key = branch_score_cache_key(branch, params; charge_cutoff, q_points, tc_points, max_rounds)
    haskey(BRANCH_SCORE_CACHE, cache_key) && return BRANCH_SCORE_CACHE[cache_key]

    candidate_snapshot = branch_candidate_snapshot(base_snapshot, branch, params)
    system = local_exact_system(candidate_snapshot; branch = branch)
    axes = branch_flux_axes(branch; q_points, tc_points)
    scan = adaptive_qubit_excited_scan(
        system,
        candidate_snapshot,
        axes.q_fluxes,
        axes.tc_fluxes,
        branch_init_levels(branch),
        params["t_gate"];
        branch = branch,
        charge_cutoff = charge_cutoff,
        max_rounds = max_rounds,
    )

    x_sorted, y_sorted, matrix_sorted = ordered_heatmap_axes(scan.x_values, scan.y_values, scan.matrix)
    best = scan.best_point
    stripe_contrast = size(matrix_sorted, 1) > 1 ? maximum(abs.(diff(matrix_sorted; dims = 1))) : 0.0

    q_probe_lo = max(first(scan.q_fluxes), best.q_flux - 0.02)
    q_probe_hi = min(last(scan.q_fluxes), best.q_flux + 0.02)
    q_probe = if q_probe_hi > q_probe_lo
        collect(range(q_probe_lo, q_probe_hi; length = 15))
    else
        collect(range(first(scan.q_fluxes), last(scan.q_fluxes); length = 15))
    end
    pair_a, pair_b = branch_pairs(branch)
    gap_rows = adiabatic_fan_rows(
        system,
        candidate_snapshot,
        q_probe,
        best.tc_flux;
        branch_pair_a = pair_a,
        branch_pair_b = pair_b,
        which = branch,
        branch = branch,
        charge_cutoff = charge_cutoff,
    )
    gap_pair = choose_highlighted_pair(gap_rows)
    gap_summary = adiabatic_gap_summary(gap_rows, gap_pair)

    edge_deficit = (best.q_edge == :interior ? 0.0 : 1.0) + (best.tc_edge == :interior ? 0.0 : 1.0)
    time_penalty = abs(params["t_gate"] - branch_default_t_gate(branch)) / 120.0
    support_penalty = max(0.0, 1.2 - min(gap_summary.support_score, 1.2))
    contrast_bonus = min(stripe_contrast, 0.25)
    value_penalty = 10.0 * best.value
    interior_penalty = 100.0 * edge_deficit
    loss = interior_penalty + value_penalty + 0.5 * time_penalty + 0.4 * support_penalty - 1.5 * contrast_bonus

    result = (
        params = copy_branch_params(params),
        loss = loss,
        best_point = best,
        stripe_contrast = stripe_contrast,
        scan_rounds = scan.rounds,
        gap_ghz = gap_summary.gap_ghz,
        gap_support = gap_summary.support_score,
        edge_deficit = edge_deficit,
        q_edge = best.q_edge,
        tc_edge = best.tc_edge,
        sampled_x = x_sorted,
        sampled_y = y_sorted,
    )
    BRANCH_SCORE_CACHE[cache_key] = result
    return result
end

function update_topk!(rows, candidate; k)
    push!(rows, candidate)
    sort!(rows, by = row -> row.loss)
    length(rows) > k && resize!(rows, k)
    return rows
end

function branch_candidate_key(row)
    params = row.params
    return (
        round(Int, params["EJmax"] * 1000),
        round(Int, params["EC"] * 1_000_000),
        round(Int, params["asymmetry"] * 10_000),
        round(Int, params["beta_qc"] * 10_000),
        round(Int, params["t_gate"] * 100),
        round(Int, params["beta_cr"] * 10_000),
        round(Int, params["tc_flux_offset"] * 10_000),
    )
end

function dedup_branch_rows(rows; keep)
    unique_rows = []
    seen = Set{NTuple{7, Int}}()
    for row in sort(rows, by = item -> item.loss)
        key = branch_candidate_key(row)
        key in seen && continue
        push!(seen, key)
        push!(unique_rows, row)
        length(unique_rows) >= keep && break
    end
    return unique_rows
end

function branch_inner_search(base_snapshot, branch::Symbol, beta_cr, EJmax, asymmetry, EC, tc_flux_offset, grids; keep = 3)
    rows = []
    for beta_qc in grids.beta_qc
        for t_gate in grids.t_gate
            params = Dict(
                "EJmax" => Float64(EJmax),
                "EC" => Float64(EC),
                "asymmetry" => Float64(asymmetry),
                "beta_qc" => Float64(beta_qc),
                "beta_cr" => Float64(beta_cr),
                "t_gate" => Float64(t_gate),
                "tc_flux_offset" => Float64(tc_flux_offset),
            )
            candidate = branch_score_summary(base_snapshot, branch, params)
            update_topk!(rows, candidate; k = keep)
        end
    end
    return rows
end

function branch_coordinate_descent(base_snapshot, branch::Symbol, seed)
    steps = Dict(
        "beta_qc" => [0.002, 0.001],
        "t_gate" => [20.0, 10.0, 5.0],
        "EJmax" => [2.0, 1.0, 0.5],
        "asymmetry" => [0.02, 0.01],
        "tc_flux_offset" => [0.02, 0.01],
    )
    best = branch_score_summary(base_snapshot, branch, copy_branch_params(seed.params))
    fields = ["tc_flux_offset", "beta_qc", "t_gate", "EJmax", "asymmetry"]

    for stage in 1:3
        improved = true
        while improved
            improved = false
            for field in fields
                stage > length(steps[field]) && continue
                for direction in (-1.0, 1.0)
                    trial_params = copy_branch_params(best.params)
                    trial_params[field] = clamp_branch_param(branch, field, trial_params[field] + direction * steps[field][stage])
                    trial = branch_score_summary(base_snapshot, branch, trial_params)
                    if trial.loss + 1e-10 < best.loss
                        best = trial
                        improved = true
                    end
                end
            end
        end
    end

    return best
end

function run_branch_search(base_snapshot, branch::Symbol, beta_cr, grids; keep = 5, seeds_per_slice = 2, multistart = 4, seed_pool = 12)
    seeds = []
    EC = base_snapshot.devices[branch_tc_key(branch)]["EC"]

    for EJmax in grids.EJmax
        for asymmetry in grids.asymmetry
            for tc_flux_offset in grids.tc_flux_offset
                for seed in branch_inner_search(base_snapshot, branch, beta_cr, EJmax, asymmetry, EC, tc_flux_offset, grids; keep = seeds_per_slice)
                    update_topk!(seeds, seed; k = seed_pool)
                end
            end
        end
    end

    refined = []
    for seed in first(dedup_branch_rows(seeds; keep = multistart), min(multistart, length(seeds)))
        update_topk!(refined, branch_coordinate_descent(base_snapshot, branch, seed); k = keep * 2)
    end

    return dedup_branch_rows(refined; keep)
end

function joint_candidate_key(row)
    move = row.params["move"]
    cz = row.params["cz"]
    return (
        round(Int, row.params["beta_cr"] * 10_000),
        round(Int, move["EJmax"] * 1000),
        round(Int, move["asymmetry"] * 10_000),
        round(Int, move["beta_qc"] * 10_000),
        round(Int, move["t_gate"] * 100),
        round(Int, move["tc_flux_offset"] * 10_000),
        round(Int, cz["EJmax"] * 1000),
        round(Int, cz["asymmetry"] * 10_000),
        round(Int, cz["beta_qc"] * 10_000),
        round(Int, cz["t_gate"] * 100),
        round(Int, cz["tc_flux_offset"] * 10_000),
    )
end

function dedup_joint_rows(rows; keep)
    unique_rows = []
    seen = Set{NTuple{11, Int}}()
    for row in sort(rows, by = item -> item.loss)
        key = joint_candidate_key(row)
        key in seen && continue
        push!(seen, key)
        push!(unique_rows, row)
        length(unique_rows) >= keep && break
    end
    return unique_rows
end

function joint_loss_summary(move_row, cz_row)
    move_edge_deficit = (move_row.q_edge == :interior ? 0.0 : 1.0) + (move_row.tc_edge == :interior ? 0.0 : 1.0)
    cz_edge_deficit = (cz_row.q_edge == :interior ? 0.0 : 1.0) + (cz_row.tc_edge == :interior ? 0.0 : 1.0)
    interior_penalty = 1000.0 * (move_edge_deficit + cz_edge_deficit)
    basin_penalty = 10.0 * (move_row.best_point.value + cz_row.best_point.value)
    stripe_bonus = min(move_row.stripe_contrast, 0.25) + min(cz_row.stripe_contrast, 0.25)
    time_penalty = 0.5 * (
        abs(move_row.params["t_gate"] - FIG2_DEFAULT_MOVE_T_GATE) / 120.0 +
        abs(cz_row.params["t_gate"] - FIG2_DEFAULT_CZ_T_GATE) / 120.0
    )
    imbalance_penalty = 0.25 * abs(move_row.loss - cz_row.loss)
    loss = interior_penalty + basin_penalty + time_penalty + imbalance_penalty - 1.5 * stripe_bonus
    return (
        loss = loss,
        interior_penalty = interior_penalty,
        basin_penalty = basin_penalty,
        stripe_bonus = stripe_bonus,
        time_penalty = time_penalty,
        imbalance_penalty = imbalance_penalty,
    )
end

function joint_shortlist(base_snapshot, working_snapshot, gate_times; profile = RETUNE_PROFILE, shortlist_size = RETUNE_SHORTLIST_SIZE)
    settings = search_profile_settings(profile)
    move_grids = branch_grids(working_snapshot, gate_times, :move, profile)
    cz_grids = branch_grids(working_snapshot, gate_times, :cz, profile)
    rows = []

    for beta_cr in beta_cr_grid(working_snapshot, profile)
        move_rows = run_branch_search(base_snapshot, :move, beta_cr, move_grids; keep = settings.branch_keep, seeds_per_slice = settings.seeds_per_slice, multistart = settings.multistart, seed_pool = settings.seed_pool)
        cz_rows = run_branch_search(base_snapshot, :cz, beta_cr, cz_grids; keep = settings.branch_keep, seeds_per_slice = settings.seeds_per_slice, multistart = settings.multistart, seed_pool = settings.seed_pool)

        for move_row in move_rows
            for cz_row in cz_rows
                params = Dict(
                    "beta_cr" => beta_cr,
                    "move" => copy_branch_params(move_row.params),
                    "cz" => copy_branch_params(cz_row.params),
                )
                summary = joint_loss_summary(move_row, cz_row)
                push!(rows, (
                    loss = summary.loss,
                    params = params,
                    move = move_row,
                    cz = cz_row,
                    joint = summary,
                ))
            end
        end
    end

    return dedup_joint_rows(rows; keep = shortlist_size)
end

function joint_candidate_snapshot(base_snapshot, params)
    device_overrides = Dict(
        :TC1 => Dict(
            "EJmax" => params["move"]["EJmax"],
            "EC" => params["move"]["EC"],
            "flux" => params["move"]["tc_flux_offset"],
            "asymmetry" => params["move"]["asymmetry"],
        ),
        :TC2 => Dict(
            "EJmax" => params["cz"]["EJmax"],
            "EC" => params["cz"]["EC"],
            "flux" => params["cz"]["tc_flux_offset"],
            "asymmetry" => params["cz"]["asymmetry"],
        ),
    )
    target_overrides = Dict(
        "beta_qc_qb1" => params["move"]["beta_qc"],
        "beta_qc_qb2" => params["cz"]["beta_qc"],
        "beta_cr" => params["beta_cr"],
        "fig2_move_t_gate_ns" => params["move"]["t_gate"],
        "fig2_cz_t_gate_ns" => params["cz"]["t_gate"],
    )
    return snapshot_with_overrides(base_snapshot; device_overrides, target_overrides)
end

function joint_candidate_cache_key(params)
    return (
        round(Int, params["beta_cr"] * 10_000),
        round(Int, params["move"]["EJmax"] * 1000),
        round(Int, params["move"]["EC"] * 1_000_000),
        round(Int, params["move"]["asymmetry"] * 10_000),
        round(Int, params["move"]["beta_qc"] * 10_000),
        round(Int, params["move"]["t_gate"] * 100),
        round(Int, params["move"]["tc_flux_offset"] * 10_000),
        round(Int, params["cz"]["EJmax"] * 1000),
        round(Int, params["cz"]["EC"] * 1_000_000),
        round(Int, params["cz"]["asymmetry"] * 10_000),
        round(Int, params["cz"]["beta_qc"] * 10_000),
        round(Int, params["cz"]["t_gate"] * 100),
        round(Int, params["cz"]["tc_flux_offset"] * 10_000),
    )
end

function evaluate_joint_candidate(base_snapshot, params)
    cache_key = joint_candidate_cache_key(params)
    haskey(JOINT_CANDIDATE_CACHE, cache_key) && return JOINT_CANDIDATE_CACHE[cache_key]

    snapshot = joint_candidate_snapshot(base_snapshot, params)
    move = branch_score_summary(base_snapshot, :move, merge(copy_branch_params(params["move"]), Dict("beta_cr" => params["beta_cr"])))
    cz = branch_score_summary(base_snapshot, :cz, merge(copy_branch_params(params["cz"]), Dict("beta_cr" => params["beta_cr"])))
    summary = joint_loss_summary(move, cz)
    result = (loss = summary.loss, params = copy_joint_params(params), move = move, cz = cz, snapshot = snapshot, joint = summary)
    JOINT_CANDIDATE_CACHE[cache_key] = result
    return result
end

function refine_joint_candidate(base_snapshot, seed)
    steps = Dict(
        "beta_qc" => [0.002, 0.001],
        "t_gate" => [20.0, 10.0, 5.0],
        "EJmax" => [2.0, 1.0, 0.5],
        "beta_cr" => [0.002, 0.001],
        "asymmetry" => [0.02, 0.01],
        "tc_flux_offset" => [0.02, 0.01],
    )
    best = evaluate_joint_candidate(base_snapshot, seed.params)
    order = [
        (:move, "tc_flux_offset"),
        (:cz, "tc_flux_offset"),
        (:move, "beta_qc"),
        (:cz, "beta_qc"),
        (:move, "t_gate"),
        (:cz, "t_gate"),
        (:move, "EJmax"),
        (:cz, "EJmax"),
        (:shared, "beta_cr"),
        (:move, "asymmetry"),
        (:cz, "asymmetry"),
    ]

    for stage in 1:3
        improved = true
        while improved
            improved = false
            for (scope, field) in order
                stage > length(steps[field]) && continue
                for direction in (-1.0, 1.0)
                    trial_params = copy_joint_params(best.params)
                    if scope == :shared
                        trial_params["beta_cr"] = clamp_beta_cr(trial_params["beta_cr"] + direction * steps[field][stage])
                    else
                        branch_name = String(scope)
                        trial_params[branch_name][field] = clamp_branch_param(scope, field, trial_params[branch_name][field] + direction * steps[field][stage])
                    end
                    trial = evaluate_joint_candidate(base_snapshot, trial_params)
                    if trial.loss + 1e-10 < best.loss
                        best = trial
                        improved = true
                    end
                end
            end
        end
    end

    return best
end

function fallback_ec_refine(base_snapshot, seed)
    best = seed
    for branch in (:move, :cz)
        branch_name = String(branch)
        for EC in FALLBACK_EC_GRID
            trial_params = copy_joint_params(best.params)
            trial_params[branch_name]["EC"] = Float64(EC)
            trial = evaluate_joint_candidate(base_snapshot, trial_params)
            if trial.loss + 1e-10 < best.loss
                best = trial
            end
        end
    end
    return best
end

function render_branch_panel(candidate_snapshot, branch::Symbol, t_gate; charge_cutoff = RETUNE_CHARGE_CUTOFF)
    system = local_exact_system(candidate_snapshot; branch = branch)
    axes = branch_flux_axes(branch; q_points = RETUNE_FINE_Q_POINTS, tc_points = RETUNE_FINE_TC_POINTS)
    scan = adaptive_qubit_excited_scan(
        system,
        candidate_snapshot,
        axes.q_fluxes,
        axes.tc_fluxes,
        branch_init_levels(branch),
        t_gate;
        branch = branch,
        charge_cutoff = charge_cutoff,
        max_rounds = 4,
    )
    x_sorted, y_sorted, matrix_sorted = ordered_heatmap_axes(scan.x_values, scan.y_values, scan.matrix)
    best_point = scan.best_point
    span = branch_span(branch)
    display = centered_span_window(best_point, x_sorted, y_sorted; delta_q_ghz = span.delta_q_ghz, delta_tc_ghz = span.delta_tc_ghz)
    return (
        x = x_sorted,
        y = y_sorted,
        matrix = matrix_sorted,
        best_point = best_point,
        display = display,
        rounds = scan.rounds,
    )
end

function isolated_tc_metrics(snapshot, branch::Symbol; charge_cutoff = 10, tc_ncut = 13)
    values, _ = local_eigenbundle(local_tc(snapshot, 0.0; branch = branch, ncut = tc_ncut); charge_cutoff = charge_cutoff)
    f01 = values[2] - values[1]
    alpha = (values[3] - values[2]) - f01
    return (f01_ghz = f01, anharmonicity_ghz = alpha)
end

function candidate_params_from_snapshot(snapshot; move_default = FIG2_DEFAULT_MOVE_T_GATE, cz_default = FIG2_DEFAULT_CZ_T_GATE)
    gate_times = fig2_gate_times(snapshot; move_default, cz_default)
    return Dict(
        "beta_cr" => snapshot.targets["beta_cr"],
        "move" => Dict(
            "EJmax" => snapshot.devices[:TC1]["EJmax"],
            "EC" => snapshot.devices[:TC1]["EC"],
            "asymmetry" => snapshot.devices[:TC1]["asymmetry"],
            "beta_qc" => snapshot.targets["beta_qc_qb1"],
            "t_gate" => gate_times.move,
            "tc_flux_offset" => snapshot.devices[:TC1]["flux"],
        ),
        "cz" => Dict(
            "EJmax" => snapshot.devices[:TC2]["EJmax"],
            "EC" => snapshot.devices[:TC2]["EC"],
            "asymmetry" => snapshot.devices[:TC2]["asymmetry"],
            "beta_qc" => snapshot.targets["beta_qc_qb2"],
            "t_gate" => gate_times.cz,
            "tc_flux_offset" => snapshot.devices[:TC2]["flux"],
        ),
    )
end

function overlay_dict_from_candidate(base_snapshot, candidate)
    snapshot = joint_candidate_snapshot(base_snapshot, candidate.params)
    tc1_metrics = isolated_tc_metrics(snapshot, :move)
    tc2_metrics = isolated_tc_metrics(snapshot, :cz)
    return Dict(
        "scope" => "Working overlay for Renger 2026 Fig. 2(e)(f) retune",
        "notes" => "Autogenerated from renger2026-figure2-ef-retune.ipynb.",
        "source_snapshot" => "output/renger2026/paper_local_priors.toml",
        "devices" => Dict(
            "TC1" => Dict(
                "EJmax" => candidate.params["move"]["EJmax"],
                "EC" => candidate.params["move"]["EC"],
                "flux" => candidate.params["move"]["tc_flux_offset"],
                "asymmetry" => candidate.params["move"]["asymmetry"],
                "f01_ghz" => tc1_metrics.f01_ghz,
                "anharmonicity_ghz" => tc1_metrics.anharmonicity_ghz,
            ),
            "TC2" => Dict(
                "EJmax" => candidate.params["cz"]["EJmax"],
                "EC" => candidate.params["cz"]["EC"],
                "flux" => candidate.params["cz"]["tc_flux_offset"],
                "asymmetry" => candidate.params["cz"]["asymmetry"],
                "f01_ghz" => tc2_metrics.f01_ghz,
                "anharmonicity_ghz" => tc2_metrics.anharmonicity_ghz,
            ),
        ),
        "targets" => Dict(
            "beta_qc_qb1" => candidate.params["move"]["beta_qc"],
            "beta_qc_qb2" => candidate.params["cz"]["beta_qc"],
            "beta_cr" => candidate.params["beta_cr"],
            "fig2_move_t_gate_ns" => candidate.params["move"]["t_gate"],
            "fig2_cz_t_gate_ns" => candidate.params["cz"]["t_gate"],
            "cr_f01_ghz" => snapshot.targets["cr_f01_ghz"],
        ),
    )
end

function write_overlay(path::AbstractString, overlay::Dict)
    open(path, "w") do io
        TOML.print(io, overlay)
    end
    return path
end


## Joint Shortlist And Gallery

The coarse search keeps one shared `beta_cr` outside and optimizes MOVE/CZ separately inside that block. The notebook then builds a top-6 joint shortlist and renders a paper-span-centered gallery around each candidate basin.


In [ ]:
shortlist = joint_shortlist(baseline_snapshot, working_snapshot, working_gate_times)

candidate_rows = [
    (
        rank = idx,
        loss = round4(row.loss),
        beta_cr = round4(row.params["beta_cr"]),
        move_EJmax = round4(row.params["move"]["EJmax"]),
        move_asymmetry = round4(row.params["move"]["asymmetry"]),
        move_beta_qc = round4(row.params["move"]["beta_qc"]),
        move_t_gate = round4(row.params["move"]["t_gate"]),
        move_tc_flux_offset = round4(row.params["move"]["tc_flux_offset"]),
        move_best = compact_namedtuple(row.move.best_point),
        move_loss = round4(row.move.loss),
        move_stripe = round4(row.move.stripe_contrast),
        move_edges = (row.move.q_edge, row.move.tc_edge),
        cz_EJmax = round4(row.params["cz"]["EJmax"]),
        cz_asymmetry = round4(row.params["cz"]["asymmetry"]),
        cz_beta_qc = round4(row.params["cz"]["beta_qc"]),
        cz_t_gate = round4(row.params["cz"]["t_gate"]),
        cz_tc_flux_offset = round4(row.params["cz"]["tc_flux_offset"]),
        cz_best = compact_namedtuple(row.cz.best_point),
        cz_loss = round4(row.cz.loss),
        cz_stripe = round4(row.cz.stripe_contrast),
        cz_edges = (row.cz.q_edge, row.cz.tc_edge),
    )
    for (idx, row) in enumerate(shortlist)
]

gallery_figure = Figure(size = (1600, max(1, length(shortlist)) * 280))
gallery_panels = NamedTuple[]
gallery_heatmaps = Any[]

for (idx, row) in enumerate(shortlist)
    candidate_snapshot = joint_candidate_snapshot(baseline_snapshot, row.params)
    move_panel = render_branch_panel(candidate_snapshot, :move, row.params["move"]["t_gate"])
    cz_panel = render_branch_panel(candidate_snapshot, :cz, row.params["cz"]["t_gate"])

    ax_move = Axis(
        gallery_figure[idx, 1];
        title = @sprintf("#%d MOVE | loss=%.3f | beta_cr=%.4f", idx, row.loss, row.params["beta_cr"]),
        xlabel = "omega_QB,01 (GHz)",
        ylabel = "omega_TC (GHz)",
    )
    hm_move = heatmap!(ax_move, move_panel.x, move_panel.y, move_panel.matrix; colorrange = (0.0, 1.0), colormap = :viridis)
    xlims!(ax_move, move_panel.display.xlims...)
    ylims!(ax_move, move_panel.display.ylims...)
    scatter!(ax_move, [move_panel.best_point.q_ghz], [move_panel.best_point.tc_ghz]; color = :white, strokecolor = :black, markersize = 10)

    ax_cz = Axis(
        gallery_figure[idx, 2];
        title = @sprintf("#%d CZ   | loss=%.3f | beta_cr=%.4f", idx, row.loss, row.params["beta_cr"]),
        xlabel = "omega_QB,01 (GHz)",
        ylabel = "omega_TC (GHz)",
    )
    hm_cz = heatmap!(ax_cz, cz_panel.x, cz_panel.y, cz_panel.matrix; colorrange = (0.0, 1.0), colormap = :viridis)
    push!(gallery_heatmaps, hm_cz)
    xlims!(ax_cz, cz_panel.display.xlims...)
    ylims!(ax_cz, cz_panel.display.ylims...)
    scatter!(ax_cz, [cz_panel.best_point.q_ghz], [cz_panel.best_point.tc_ghz]; color = :white, strokecolor = :black, markersize = 10)

    push!(gallery_panels, (
        rank = idx,
        move = (best_point = move_panel.best_point, display = move_panel.display),
        cz = (best_point = cz_panel.best_point, display = cz_panel.display),
    ))
end

Colorbar(gallery_figure[:, 3], gallery_heatmaps[end]; label = "final qubit P_e")

gallery_saved = save_figure(gallery_figure, repo_root, retune_artifact_stem("figure2ef_retune_gallery"))
retune_exports[:gallery] = gallery_saved

display(gallery_figure)
(
    shortlist = candidate_rows,
    gallery_saved = gallery_saved,
    gallery_panels = gallery_panels,
)


## Select, Refine, And Persist

The selected rank is refined with a narrow coordinate descent around the shortlist parameters. `SELECT_CANDIDATE_RANK = 1` is only a default for the first pass; the intended workflow is to inspect the gallery and change the rank before refinement. Writing back to the working overlay is optional and disabled by default.


In [ ]:
selected_seed = shortlist[SELECT_CANDIDATE_RANK]
selected_candidate = RUN_REFINEMENT ? refine_joint_candidate(baseline_snapshot, selected_seed) : evaluate_joint_candidate(baseline_snapshot, selected_seed.params)
if RUN_EC_FALLBACK && (
    selected_candidate.move.best_point.q_edge != :interior ||
    selected_candidate.move.best_point.tc_edge != :interior ||
    selected_candidate.cz.best_point.q_edge != :interior ||
    selected_candidate.cz.best_point.tc_edge != :interior
)
    selected_candidate = fallback_ec_refine(baseline_snapshot, selected_candidate)
end

selected_snapshot = joint_candidate_snapshot(baseline_snapshot, selected_candidate.params)
selected_move = render_branch_panel(selected_snapshot, :move, selected_candidate.params["move"]["t_gate"])
selected_cz = render_branch_panel(selected_snapshot, :cz, selected_candidate.params["cz"]["t_gate"])
selected_overlay = overlay_dict_from_candidate(baseline_snapshot, selected_candidate)
selected_overlay_toml = sprint(io -> TOML.print(io, selected_overlay))
current_overlay_candidate = evaluate_joint_candidate(baseline_snapshot, candidate_params_from_snapshot(working_snapshot))
selected_better_than_working = selected_candidate.loss + 1e-10 < current_overlay_candidate.loss

selected_fig = Figure(size = (1500, 600))
ax_move = Axis(selected_fig[1, 1]; title = "Selected MOVE panel", xlabel = "omega_QB,01 (GHz)", ylabel = "omega_TC (GHz)")
heatmap!(ax_move, selected_move.x, selected_move.y, selected_move.matrix; colorrange = (0.0, 1.0), colormap = :viridis)
xlims!(ax_move, selected_move.display.xlims...)
ylims!(ax_move, selected_move.display.ylims...)
scatter!(ax_move, [selected_move.best_point.q_ghz], [selected_move.best_point.tc_ghz]; color = :white, strokecolor = :black, markersize = 12)

ax_cz = Axis(selected_fig[1, 2]; title = "Selected CZ panel", xlabel = "omega_QB,01 (GHz)", ylabel = "omega_TC (GHz)")
heatmap!(ax_cz, selected_cz.x, selected_cz.y, selected_cz.matrix; colorrange = (0.0, 1.0), colormap = :viridis)
xlims!(ax_cz, selected_cz.display.xlims...)
ylims!(ax_cz, selected_cz.display.ylims...)
scatter!(ax_cz, [selected_cz.best_point.q_ghz], [selected_cz.best_point.tc_ghz]; color = :white, strokecolor = :black, markersize = 12)

selected_saved = save_figure(selected_fig, repo_root, retune_artifact_stem("figure2ef_selected_retune_preview"))
retune_exports[:selected] = selected_saved

display(selected_fig)

written_overlay = if WRITE_WORKING_OVERLAY && selected_better_than_working
    write_overlay(RETUNE_OVERLAY_PATH, selected_overlay)
else
    nothing
end

(
    selected_rank = SELECT_CANDIDATE_RANK,
    refined = RUN_REFINEMENT,
    loss = round4(selected_candidate.loss),
    move = (
        params = selected_candidate.params["move"],
        best_point = compact_namedtuple(selected_move.best_point),
    ),
    cz = (
        params = selected_candidate.params["cz"],
        best_point = compact_namedtuple(selected_cz.best_point),
    ),
    beta_cr = round4(selected_candidate.params["beta_cr"]),
    selected_saved = selected_saved,
    written_overlay = written_overlay,
    selected_better_than_working = selected_better_than_working,
    current_overlay_loss = round4(current_overlay_candidate.loss),
    overlay_toml = selected_overlay_toml,
)


## Reproduction Check

This final cell reloads the snapshot through the optional overlay path and reports the effective TC/gate-time values that the Figure 2 reproduction notebook will see.


In [ ]:
reloaded_snapshot = load_renger2026_snapshot(
    RETUNE_BASELINE_PATH;
    overlay_path = isfile(RETUNE_OVERLAY_PATH) ? RETUNE_OVERLAY_PATH : nothing,
)
reloaded_gate_times = fig2_gate_times(
    reloaded_snapshot;
    move_default = FIG2_DEFAULT_MOVE_T_GATE,
    cz_default = FIG2_DEFAULT_CZ_T_GATE,
)
reloaded_rows, reloaded_couplings = parameter_rows(
    reloaded_snapshot;
    move_t_gate_ns = reloaded_gate_times.move,
    cz_t_gate_ns = reloaded_gate_times.cz,
    charge_cutoff = RETUNE_CHARGE_CUTOFF,
)

(
    overlay_active = isfile(RETUNE_OVERLAY_PATH),
    parameter_rows = reloaded_rows,
    couplings = reloaded_couplings,
    exports = retune_exports,
)
